In [1]:
import hashlib
import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from lancedb.rerankers import ColbertReranker
import torch
import ollama
import os
import re
import json
import subprocess
import time
from collections import Counter
from tqdm.auto import tqdm                      # notebook widget when ipywidgets is installed, console bar otherwise
from codecarbon import EmissionsTracker

# Tier 1 (deterministic table gate) + Tier 2 (single-forward-pass SLM) slice-radius predictor.
# Prompts, exemplars and decision rule are the ones evaluated in ollama_dynamic_slice_length.ipynb.
from utils.dynamic_slice_prediction import DynamicSlicePredictor, OLLAMA_OPTIONS as PREDICTOR_OPTIONS

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
MAX_TOKENS = 2000
OLLAMA_MODEL_NAME = "gemma3:4b-it-qat"                 # ONE model for both the radius predictor and the summariser
SUMMARISER_MODELFILE = "Models/chunker_full_doc.Modelfile"   # the custom 'chunker_full_doc' model is just this base model + SYSTEM + temperature 0 + num_ctx 16384
SKIP_CONTEXT_FOR_K0 = True                             # k = 0 means "already a summary / boilerplate": no LLM call, the chunk is embedded as is
DOC_LIMIT = None                                       # e.g. 2 for a smoke test on the first studies; None = every study in INPUT_DIR

RUN_NAME = "ablation_doc_slice_radius_dynamic"         # new artefacts only: the fixed-radius JSONs / tables stay untouched
CHUNKS_WITH_METADATA_FILE_NAME = f"preprocessed_chunks/{RUN_NAME}.json"
TABLE_NAME = RUN_NAME
DB_PATH = "./db"
EMISSIONS_DIR = "./emissions_data"                     # codecarbon appends one row per run to emissions.csv here

INPUT_DIR = "split_documents"                          # the 25 scientific papers (Docling HybridChunker output)


_INT_PARAMS = {"num_ctx", "num_predict", "top_k", "seed", "num_gpu", "num_thread", "repeat_last_n", "num_batch", "num_keep", "mirostat"}

def load_modelfile(path):
    """(system_prompt, options) from an Ollama Modelfile, so the custom model's behaviour can be requested per call
    on the plain base model instead of loading a second 4 GB runner. The SYSTEM block is kept verbatim (Ollama does
    not trim it either); PARAMETER lines become the `options` dict of ollama.chat."""
    text = open(path, encoding="utf-8").read()
    m = re.search(r'^SYSTEM\s+"""(.*?)"""', text, flags=re.S | re.M)
    system = m.group(1) if m else None
    options = {}
    for name, value in re.findall(r"^PARAMETER\s+(\w+)\s+(.+?)\s*$", text, flags=re.M):
        value = value.strip().strip('"')
        if name in _INT_PARAMS:
            value = int(value)
        else:
            try:
                value = float(value)
            except ValueError:
                pass
        if name == "stop":
            options.setdefault("stop", []).append(value)
        else:
            options[name] = value
    return system, options

SUMMARISER_SYSTEM_PROMPT, SUMMARISER_OPTIONS = load_modelfile(SUMMARISER_MODELFILE)
# Ollama reloads a model whenever a load-time option such as num_ctx changes. Asking for the summariser's context
# window in the predictor's calls too keeps a single runner resident for the whole run (no unload/reload per phase).
PREDICTOR_OPTIONS = {**PREDICTOR_OPTIONS, "num_ctx": SUMMARISER_OPTIONS["num_ctx"]}
print(f"summariser = {OLLAMA_MODEL_NAME} + options {SUMMARISER_OPTIONS} + system prompt ({len(SUMMARISER_SYSTEM_PROMPT)} chars) | predictor options {PREDICTOR_OPTIONS}")

study_names = sorted(f for f in os.listdir(INPUT_DIR) if f.endswith('.json'))
processed_chunks=[]
try:
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        processed_chunks = json.load(f)
except FileNotFoundError:
    print(f"No existing {CHUNKS_WITH_METADATA_FILE_NAME} file found, starting fresh.")


chunks_with_metadata = processed_chunks.copy()
processed_studies = set(chunk["document"] for chunk in processed_chunks)

study_names = [f for f in study_names if f not in processed_studies]
if DOC_LIMIT:
    study_names = study_names[:DOC_LIMIT]
print(f"Found {len(processed_studies)} studies which are already processed.\nStudies which STILL need to be processed: {len(study_names)}:\n{study_names}...")

# Note: These results are from chunks created by doclign's layout aware hybrid chunker, which is more aggressive in splitting text into smaller chunks.
# k=4 --> 28m11s
# k=3 --> 22m41s
# k=2 --> 17m28s
# k=1 --> 12m08s
# k=0 --> 7m28s
# k=dynamic --> 9m06s
# Note: These results are from AutoTokenizer-based chunking, which is less aggressive, resulting in fewer chunks per document, with 2000 tokens per chunks and a 200 token overlap. LIMITATION: Due to the relatively short documents, using 2000 tokens meant that having a k=3 sliding window often included the entire document, and thus had functionally no difference between k=4 and the anthropic baseline.
# k=4 --> 9m53s
# k=3 --> 10m58s
# k=2 --> 11m01s
# k=1 --> 8m09s
# k=0 --> 4m53

summariser = gemma3:4b-it-qat + options {'temperature': 0.0, 'num_ctx': 16384} + system prompt (789 chars) | predictor options {'temperature': 0.0, 'num_predict': 1, 'top_p': 1.0, 'num_ctx': 16384}
No existing preprocessed_chunks/ablation_doc_slice_radius_dynamic.json file found, starting fresh.
Found 0 studies which are already processed.
Studies which STILL need to be processed: 25:
['A_Conceptual_Framework_and_Recommendations_for_Open_Data_and_Artifacts_in_Empirical_Software_Engineering.pdf.json', 'A_Feature_Fusion_Based_Indicator_for_Training-Free_Neural_Architecture_Search.pdf.json', 'A_Hybrid_Gaze_Distance_Estimation_via_Cross-Reference_of_Vergence_and_Depth.pdf.json', 'A_Resource_Allocation_Model_Based_on_Trust_Evaluation_in_Multi-Cloud_Environments.pdf.json', 'Electron_Paramagnetic_Resonance_Study_on_28Si_Single_Crystal_for_the_Future_Realization_of_the_Kilogram.pdf.json', 'Probabilistic_Artificial_Neural_Network_for_Line-Edge-Roughness-Induced_Random_Variation_in_FinFET.pdf.js

In [2]:
# Radius predictor on the SAME model and context window as the summariser -> one resident runner, no swapping.
predictor = DynamicSlicePredictor(model=OLLAMA_MODEL_NAME, options=PREDICTOR_OPTIONS)

os.makedirs(EMISSIONS_DIR, exist_ok=True)      # codecarbon refuses to start if the output folder is missing
tracker = EmissionsTracker(
        project_name=RUN_NAME,
        measure_power_secs=1,
        output_dir=EMISSIONS_DIR,
        log_level="error"
    )

radius_counter = Counter()
summariser_loads = 0                            # summariser calls that had to (re)load the model; expected 0 (the predictor loads it first), each extra one is a runner swap
t_start = time.perf_counter()

with tracker:
	for source in tqdm(study_names, desc="Chunking documents..."):
		with open(os.path.join(INPUT_DIR, source), "r", encoding="utf-8") as f:
			chunks = json.load(f)

		# ---- Phase 1: suggest a slice radius for EVERY chunk of the document, sequentially, before any summary is written.
		#      k = 3 comes from the table gate without a model call; everything else is one num_predict=1 forward pass.
		suggestions = []
		for chunk in tqdm(chunks, desc=f"Predicting slice radius for {source[:20]}...", leave=False):
			k, tier, _scores, _evidence = predictor.predict_with_details(chunk["text"])
			suggestions.append((k, tier))
		radius_counter.update(k for k, _ in suggestions)

		# ---- Phase 2: the regular summary generation, now with the per-chunk radius instead of a fixed one.
		for chunk_index, chunk in enumerate(tqdm(chunks, desc=f"Adding context for chunks of {source[:20]}...", leave=False)):
			doc_slice_radius, slice_tier = suggestions[chunk_index]

			if doc_slice_radius == 0 and SKIP_CONTEXT_FOR_K0:
				# Abstract, conclusion, references, acknowledgements, ...: already self-describing, nothing to contextualise.
				context = ""
				text_to_embed = chunks[chunk_index]['text']
			else:
				doc_slice = ""

				start_index_original = chunk_index - doc_slice_radius
				start_index_truncated = max(0, start_index_original) # Avoid index out of bounds

				end_index_original = chunk_index + doc_slice_radius
				end_index_truncated = min(len(chunks)-1, end_index_original)

				if start_index_original < 0: # We are at the start of the document, so we need to add more chunks at the end
					end_index_truncated = min(len(chunks)-1, end_index_truncated + abs(start_index_original))
				if end_index_original > len(chunks)-1: # We are at the end of the document, so we need to add more chunks at the start
					start_index_truncated = max(0, start_index_truncated - abs(end_index_original - end_index_truncated))

				for i in range(start_index_truncated, end_index_truncated + 1):
					doc_slice += " " + chunks[i]["text"]

				doc_slice = "FULL DOCUMENT:\n" + doc_slice
				ollama_prompt = f"CHUNK:\n{chunks[chunk_index]['text']}"
				history =  [{'role': 'user', 'content': doc_slice}, {'role': 'user', 'content': ollama_prompt}]

				# Same rendered prompt as the 'chunker_full_doc' model: its SYSTEM prompt goes in as a system message,
				# its PARAMETERs (temperature 0, num_ctx 16384) as per-call options, on the shared base model.
				response = ollama.chat(
					model=OLLAMA_MODEL_NAME,
					messages=[{'role': 'system', 'content': SUMMARISER_SYSTEM_PROMPT}] + history,
					options=SUMMARISER_OPTIONS,
				)
				summariser_loads += response['load_duration'] > 1e9    # > 1 s of load time means the runner was (re)created
				context = response['message']['content']
				text_to_embed = context + "\n\n" + chunks[chunk_index]['text']

			id = hashlib.sha256(chunks[chunk_index]['text'].encode()).hexdigest()
			chunks_with_metadata.append({'text': text_to_embed, 'original_text':chunks[chunk_index]['text'], 'context':context, 'document':source, 'id': id,
										 'doc_slice_radius': doc_slice_radius, 'slice_tier': slice_tier})

	subprocess.run(["ollama", "stop", OLLAMA_MODEL_NAME], check=False)   # free the GPU once the whole run is over

elapsed = time.perf_counter() - t_start
print(f"Processed {len(study_names)} studies, {sum(radius_counter.values())} chunks in {elapsed/60:.1f} min")
print(f"Suggested radius distribution: {dict(sorted(radius_counter.items()))}  (k=0 chunks skipped the summariser: {SKIP_CONTEXT_FOR_K0})")
print(f"Summariser calls that (re)loaded the model: {summariser_loads}  (expected 0: the predictor loads the shared runner once; anything else means a model swap)")
emissions_data = getattr(tracker, "final_emissions_data", None)
if emissions_data is not None:
	print(f"codecarbon: {emissions_data.emissions:.6f} kg CO2eq, {emissions_data.energy_consumed:.6f} kWh over {emissions_data.duration:.0f} s -> {EMISSIONS_DIR}/emissions.csv")

[codecarbon WARNING @ 22:23:27] Multiple instances of codecarbon are allowed to run at the same time.


Chunking documents...:   0%|          | 0/25 [00:00<?, ?it/s]

Predicting slice radius for A_Conceptual_Framewo...:   0%|          | 0/21 [00:00<?, ?it/s]

Adding context for chunks of A_Conceptual_Framewo...:   0%|          | 0/21 [00:00<?, ?it/s]

Predicting slice radius for A_Feature_Fusion_Bas...:   0%|          | 0/26 [00:00<?, ?it/s]

...

Adding context for chunks of srep05215.pdf.json...:   0%|          | 0/15 [00:00<?, ?it/s]

Predicting slice radius for srep45325.pdf.json...:   0%|          | 0/8 [00:00<?, ?it/s]

Adding context for chunks of srep45325.pdf.json...:   0%|          | 0/8 [00:00<?, ?it/s]

Processed 25 studies, 382 chunks in 9.2 min
Suggested radius distribution: {0: 109, 1: 53, 2: 161, 3: 59}  (k=0 chunks skipped the summariser: True)
Summariser calls that (re)loaded the model: 0  (expected 0: the predictor loads the shared runner once; anything else means a model swap)
codecarbon: 0.002918 kg CO2eq, 0.013790 kWh over 549 s -> ./emissions_data/emissions.csv


⠙ 

In [3]:
chunks_with_metadata[0]

{'text': 'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\nEmma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\nMartin Höst martin.host@mau.se Malmö University Malmö, Sweden',
 'original_text': 'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\nEmma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\nMartin Höst martin.host@mau.se Malmö University Malmö, Sweden',
 'context': '',
 'document': 'A_Conceptual_Framework_and_Recommendations_for_Open_Data_and_Artifacts_in_Empirical_Software_Engineering.pdf.json',
 'id': '0cf3ba2977c608428dc3049ab7baa19a26e3afac77659cd7b1ff6d1af425c704',
 'doc_slice_radius': 0,
 'slice_tier': 'slm'}

In [4]:
# Save the the processed chunks in case VectorDB upload goes wrong.
# Luckily since this is a notebook, if the chunking is interrupted, we can still save the partial results here.
# Append new chunks to the existing file if it exists, otherwise create it
if os.path.exists(CHUNKS_WITH_METADATA_FILE_NAME):
    print(f"Appending to existing {CHUNKS_WITH_METADATA_FILE_NAME} file.")
    with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
        existing_data = json.load(f)
    # Avoid duplicate entries by id
    existing_ids = {chunk['id'] for chunk in existing_data}
    new_chunks = [chunk for chunk in chunks_with_metadata if chunk['id'] not in existing_ids]
    chunks_with_metadata = existing_data + new_chunks

with open(CHUNKS_WITH_METADATA_FILE_NAME, "w", encoding="utf-8") as f:
    json.dump(chunks_with_metadata, f, ensure_ascii=False, indent=2)

print(f"Results saved to {CHUNKS_WITH_METADATA_FILE_NAME}")

Results saved to preprocessed_chunks/ablation_doc_slice_radius_dynamic.json


In [5]:
from devtools import debug
registry = get_registry()
hf = registry.get("huggingface").create(name=EMBEDDING_MODEL_NAME, trust_remote_code=True, device="cuda" if torch.cuda.is_available() else "cpu")


# Define model
class MyDocument(LanceModel):
    text: str = hf.SourceField()
    vector: Vector(hf.ndims()) = hf.VectorField()
    original_text: str
    context: str
    document: str
    id: str  # Unique identifier for the chunk
    doc_slice_radius: int  # Radius suggested by utils/dynamic_slice_prediction.py (0..3)
    slice_tier: str        # "table" (deterministic gate) or "slm" (single forward pass)


db = lancedb.connect(DB_PATH)
db.create_table(TABLE_NAME, schema=MyDocument, mode="overwrite") # Uncomment this line when running this cell for the first time
table = db.open_table(TABLE_NAME)

# Upload in batches with progress bar
with open(CHUNKS_WITH_METADATA_FILE_NAME, "r", encoding="utf-8") as f:
    chunks_with_metadata = json.load(f)
    debug(chunks_with_metadata[0])

batch_size = 100
for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Uploading chunks to VectorDB"):
    batch = chunks_with_metadata[i:i+batch_size]
    table.add(batch)

table.create_scalar_index("id", replace=True) # Index based on the chunk's id, used to manually prevent duplicates

table.create_fts_index("text", replace=True) # Used by the reranker as well as the hybrid search's BM25 index
table.wait_for_index(["text_idx"])  # Wait for the indexing to finish

<All keys matched successfully>


/tmp/ipykernel_9287/4215133382.py:25 <module>
    chunks_with_metadata[0]: {
        'text': (
            'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\n'
            'Emma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\n'
            'Martin Höst martin.host@mau.se Malmö University Malmö, Sweden'
        ),
        'original_text': (
            'Per Runeson per.runeson@cs.lth.se Lund University Lund, Sweden\n'
            'Emma Söderberg emma.soderberg@cs.lth.se Lund University Lund, Sweden\n'
            'Martin Höst martin.host@mau.se Malmö University Malmö, Sweden'
        ),
        'context': '',
        'document': (
            'A_Conceptual_Framework_and_Recommendations_for_Open_Data_and_Artifacts_in_Empirical_Software_Engineering.'
            'pdf.json'
        ),
        'id': '0cf3ba2977c608428dc3049ab7baa19a26e3afac77659cd7b1ff6d1af425c704',
        'doc_slice_radius': 0,
        'slice_tier': 'slm',
    } (dict) len=7


Uploading chunks to VectorDB:   0%|          | 0/4 [00:00<?, ?it/s]

<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>
<All keys matched successfully>
